In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

In [2]:
PATH_MOVIELENS = os.path.join('..','dataset','MovieLens')

In [175]:
movie = pd.read_csv(os.path.join(PATH_MOVIELENS,'movies.csv'))
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
ratings = pd.read_csv(os.path.join(PATH_MOVIELENS,'ratings.csv'))
ratings.head()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [5]:
ratings.shape

(25000095, 4)

In [6]:
ratings['movieId'].nunique()

59047

In [7]:
ratings['userId'].nunique()

162541

In [8]:
ratings['rating'].describe()

count    2.500010e+07
mean     3.533854e+00
std      1.060744e+00
min      5.000000e-01
25%      3.000000e+00
50%      3.500000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64

In [9]:
ratings.isna().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [10]:
ratings = ratings.drop(columns='timestamp')

### SVD model

In [11]:
from surprise import SVD,Dataset, Reader

In [23]:
rating_sample = ratings.sample(n = 2500000,random_state=45)

In [24]:
# create reader
reader = Reader(
        rating_scale=(0.5,5)
        )


In [25]:
data = Dataset.load_from_df(
    df = rating_sample[['userId','movieId','rating']],
    reader=reader
    )

In [26]:
from surprise.model_selection import train_test_split

trainset,testset = train_test_split(data,test_size=0.2,random_state=42)

In [27]:
model = SVD(n_factors=100,n_epochs=20,lr_all = 0.005,reg_all=0.02,verbose=True)

In [28]:
model.fit(trainset)

Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19


In [29]:
predictions = model.test(testset)

In [31]:
predictions[0]

Prediction(uid=68674, iid=165103, r_ui=4.0, est=np.float64(3.587770641106246), details={'was_impossible': False})

In [33]:
from surprise import accuracy

print('rmse : ',accuracy.rmse(predictions))
print('mse : ',accuracy.mse(predictions))
print('mae : ',accuracy.mae(predictions))

RMSE: 0.9014
rmse :  0.9013943358537364
MSE: 0.8125
mse :  0.8125117487091985
MAE:  0.6911
mae :  0.6910522704368788


In [34]:
model.predict(uid=1,iid=54)

Prediction(uid=1, iid=54, r_ui=None, est=np.float64(2.554150861071789), details={'was_impossible': False})

In [71]:
testset_df =pd.DataFrame(columns=['userId','movieId','rating'],data=(np.array(testset)))
testset_df['userId'] = testset_df['userId'].astype('int')
testset_df['movieId'] = testset_df['movieId'].astype('int64')



In [79]:

testset_df.query('userId ==5').sort_values(by='rating' ,ascending=False)

,userId,movieId,rating
83293,5,296,4.0
378316,5,1291,4.0


In [81]:
model.predict(uid=5,iid=296)

Prediction(uid=5, iid=296, r_ui=None, est=np.float64(4.563864836418438), details={'was_impossible': False})

In [173]:
# save model
import pickle

pickle.dump(model,open('../artifact/svd_model.pkl','wb'))

### prediction demo

In [174]:
user_id = 132612
watched_movies = set(ratings[ratings['userId']==user_id]['movieId'])

all_movies = set(movies['movieId'])

unwatched_movies =  all_movies-watched_movies

predictions = []
for movie in unwatched_movies:
    pred_rating = model.predict(user_id,movie).est
    predictions.append([movie,pred_rating])
   

# sort the predicted rating by highly rated movie prediction
predictions.sort(key=lambda x : x[1],reverse=True)

movie_lookup = dict(zip(movies['movieId'],movies['title']))

for movieId, rating in predictions[:10]:
    print(movie_lookup[movieId],' : ',rating)

Over the Garden Wall (2013)  :  4.639698104441924
Planet Earth II (2016)  :  4.61108453355266
Planet Earth (2006)  :  4.5293637119999035
Tokyo Story (Tôkyô monogatari) (1953)  :  4.472514675940606
City of God (Cidade de Deus) (2002)  :  4.464637733385593
Ugetsu (Ugetsu monogatari) (1953)  :  4.425246895506177
Band of Brothers (2001)  :  4.423361207594042
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)  :  4.421838401904274
Blue Planet II (2017)  :  4.405256194226321
Fargo (1996)  :  4.404046807468935
